# Task 3: Granulometry Benchmarking — Qwen2.5-VL-3B (Base Model)

Benchmark the un-fine-tuned model on 108 test images of concrete aggregate.
The model must classify each image by max particle size (8/16/32mm) and grading (coarse/medium/fine).

This establishes the baseline that Task 4 (LoRA fine-tuning) should improve.

In [ ]:
import os, json, re, time, torch
import numpy as np
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Config

In [ ]:
# Adjust these paths to match your compute instance
TEST_DIR = '../../datasets/granulometry/test'
MANIFEST = '../../datasets/granulometry/test_manifest.json'
IMG_SIZE = 500

PROMPT = '''This image shows concrete aggregate particles photographed from above.
The image resolution is 8 pixels per mm.

Analyze the particles and respond with ONLY a JSON object (no other text):
{"max_particle_size_mm": <estimated maximum particle size as integer: 8, 16, or 32>, "grading": "<coarse, medium, or fine>"}

The values 8, 16, 32 for max_particle_size_mm and coarse, medium, fine for grading are for reference. Use the actual values based on what you observe.'''

print(f'Prompt:\n{PROMPT}')

## Load Dataset

In [ ]:
with open(MANIFEST) as f:
    manifest = json.load(f)

print(f'Total test images: {len(manifest)}')
print(f'\nClass distribution:')
from collections import Counter
cls_counts = Counter(e['class'] for e in manifest)
for cls in sorted(cls_counts):
    print(f'  {cls}: {cls_counts[cls]} images')

In [ ]:
# Preview one image from each class
classes = sorted(set(e['class'] for e in manifest))
fig, axes = plt.subplots(1, len(classes), figsize=(3*len(classes), 3))
for i, cls in enumerate(classes):
    entry = next(e for e in manifest if e['class'] == cls)
    img = Image.open(os.path.join(TEST_DIR, entry['image'])).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    axes[i].imshow(img); axes[i].set_title(f"{cls}\n{entry['grading']}, {entry['max_particle_size_mm']}mm", fontsize=9); axes[i].axis('off')
plt.suptitle('Sample from each class', fontsize=13)
plt.tight_layout(); plt.show()

## Load Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-3B-Instruct')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-3B-Instruct', torch_dtype=torch.bfloat16, device_map='auto'
)
print('Qwen2.5-VL-3B loaded.')

## Helper Functions

In [ ]:
def parse_response(raw):
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    size_m = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    grad_m = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if size_m and grad_m:
        return {'max_particle_size_mm': int(size_m.group(1)), 'grading': grad_m.group(1)}
    return None

def infer(image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':PROMPT}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
    t = time.time()
    ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    elapsed = time.time() - t
    out = processor.batch_decode(ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    return out.strip(), elapsed

print('Helpers ready.')

## Quick Test (5 images)

In [ ]:
# Test on 5 images first to make sure everything works
for entry in manifest[:5]:
    img = Image.open(os.path.join(TEST_DIR, entry['image'])).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    raw, elapsed = infer(img)
    parsed = parse_response(raw)
    print(f"{entry['class']:>3} | GT: size={entry['max_particle_size_mm']}, grading={entry['grading']}")
    print(f"     Pred: {parsed} | {elapsed:.1f}s")
    print(f"     Raw: {raw[:100]}")
    print()

## Full Benchmark (all 108 images)
This will take ~10-20 minutes depending on GPU speed.

In [ ]:
results = []
correct_size = 0
correct_grading = 0
valid_json = 0
total_time = 0

for i, entry in enumerate(manifest):
    img_path = os.path.join(TEST_DIR, entry['image'])
    if not os.path.exists(img_path):
        continue
    
    image = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    raw, elapsed = infer(image)
    total_time += elapsed
    
    parsed = parse_response(raw)
    gt_size = entry['max_particle_size_mm']
    gt_grading = entry['grading']
    
    size_ok = False
    grading_ok = False
    is_valid = parsed is not None
    
    if is_valid:
        valid_json += 1
        if parsed.get('max_particle_size_mm') == gt_size: size_ok = True; correct_size += 1
        if parsed.get('grading', '').lower() == gt_grading: grading_ok = True; correct_grading += 1
    
    results.append({
        'image': entry['image'], 'class': entry['class'],
        'gt_size': gt_size, 'gt_grading': gt_grading,
        'predicted': parsed, 'raw': raw,
        'size_correct': size_ok, 'grading_correct': grading_ok,
        'valid_json': is_valid, 'time_s': round(elapsed, 2),
    })
    
    if (i+1) % 10 == 0:
        n = i + 1
        print(f'[{n}/{len(manifest)}] Size: {correct_size}/{n} ({correct_size/n*100:.0f}%) | '
              f'Grading: {correct_grading}/{n} ({correct_grading/n*100:.0f}%) | '
              f'JSON: {valid_json}/{n} | Avg: {total_time/n:.1f}s')

print(f'\nDone. {len(results)} images processed in {total_time:.0f}s')

## Results Summary

In [ ]:
n = len(results)
both_correct = sum(1 for r in results if r['size_correct'] and r['grading_correct'])

print('=' * 60)
print('BENCHMARK RESULTS — Qwen2.5-VL-3B (Base Model)')
print('=' * 60)
print(f'Images tested:        {n}')
print(f'JSON validity:        {valid_json}/{n} ({valid_json/n*100:.1f}%)')
print(f'Size accuracy:        {correct_size}/{n} ({correct_size/n*100:.1f}%)')
print(f'Grading accuracy:     {correct_grading}/{n} ({correct_grading/n*100:.1f}%)')
print(f'Both correct:         {both_correct}/{n} ({both_correct/n*100:.1f}%)')
print(f'Avg inference time:   {total_time/n:.2f}s')

In [ ]:
# Per-class breakdown
print(f'{"Class":<6} {"Size Acc":>10} {"Grading Acc":>12} {"Both":>8}')
print('-' * 40)
class_results = defaultdict(list)
for r in results:
    class_results[r['class']].append(r)

class_data = {}
for cls in sorted(class_results):
    cr = class_results[cls]
    sc = sum(1 for r in cr if r['size_correct'])
    gc = sum(1 for r in cr if r['grading_correct'])
    bc = sum(1 for r in cr if r['size_correct'] and r['grading_correct'])
    class_data[cls] = {'size': sc/len(cr), 'grading': gc/len(cr), 'both': bc/len(cr)}
    print(f'{cls:<6} {sc:>4}/{len(cr):<4} {gc:>6}/{len(cr):<4} {bc:>4}/{len(cr)}')

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

classes = sorted(class_data.keys())
x = range(len(classes))

# Size accuracy per class
axes[0].bar(x, [class_data[c]['size']*100 for c in classes], color='steelblue')
axes[0].set_xticks(x); axes[0].set_xticklabels(classes, rotation=45)
axes[0].set_ylabel('Accuracy %'); axes[0].set_title('Max Particle Size Accuracy')
axes[0].set_ylim(0, 105); axes[0].axhline(y=correct_size/n*100, color='red', linestyle='--', label=f'Overall: {correct_size/n*100:.0f}%')
axes[0].legend()

# Grading accuracy per class
axes[1].bar(x, [class_data[c]['grading']*100 for c in classes], color='coral')
axes[1].set_xticks(x); axes[1].set_xticklabels(classes, rotation=45)
axes[1].set_ylabel('Accuracy %'); axes[1].set_title('Grading Accuracy')
axes[1].set_ylim(0, 105); axes[1].axhline(y=correct_grading/n*100, color='red', linestyle='--', label=f'Overall: {correct_grading/n*100:.0f}%')
axes[1].legend()

# Overall summary
metrics = ['JSON Valid', 'Size Acc', 'Grading Acc', 'Both Correct']
values = [valid_json/n*100, correct_size/n*100, correct_grading/n*100, both_correct/n*100]
bars = axes[2].bar(metrics, values, color=['gray', 'steelblue', 'coral', 'green'])
axes[2].set_ylabel('Percentage'); axes[2].set_title('Overall Metrics')
axes[2].set_ylim(0, 105)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.0f}%', ha='center', fontsize=11)

plt.suptitle('Task 3: Granulometry Benchmark — Qwen2.5-VL-3B (Base Model)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Show some wrong predictions
wrong = [r for r in results if not (r['size_correct'] and r['grading_correct']) and r['valid_json']]
print(f'Wrong predictions (with valid JSON): {len(wrong)}')
print()
for r in wrong[:10]:
    p = r['predicted']
    print(f"{r['class']:>3} | GT: size={r['gt_size']}, grading={r['gt_grading']} | "
          f"Pred: size={p.get('max_particle_size_mm','?')}, grading={p.get('grading','?')} | "
          f"{'Size ✗' if not r['size_correct'] else 'Size ✓'} {'Grading ✗' if not r['grading_correct'] else 'Grading ✓'}")

In [ ]:
# Save results for comparison with Task 4
with open('benchmark_results.json', 'w') as f:
    json.dump({
        'model': 'Qwen2.5-VL-3B-Instruct',
        'phase': 'base_model',
        'total_images': n,
        'json_validity_pct': round(valid_json/n*100, 1),
        'size_accuracy_pct': round(correct_size/n*100, 1),
        'grading_accuracy_pct': round(correct_grading/n*100, 1),
        'both_correct_pct': round(both_correct/n*100, 1),
        'avg_inference_time_s': round(total_time/n, 2),
        'results': results,
    }, f, indent=2)
print('Results saved to benchmark_results.json')
print('This file will be compared against Task 4 (post-fine-tuning) results.')